# EO Tutorial: xarray, Dask, and `cubo`

This notebook is a compact tutorial for **medium-resolution Earth Observation (EO)** workflows focused on **climate resilience** and **risk monitoring**.

It uses a single user-facing API — [`cubo`](https://cubo.readthedocs.io/) — to show how the same workflow can target:

- **Google Earth Engine** with `gee=True`
- **STAC catalogs** (Planetary Computer by default)
- **alternative STAC endpoints** with `stac="..."`

The notebook is designed to accompany a 90-minute tutorial with simple examples for **Africa** and **Asia**.


## Learning goals

By the end of this notebook, participants should be able to:

1. explain why **xarray** is useful for EO data cubes,
2. describe how **Dask** enables chunked and lazy processing,
3. make a basic API request with **`cubo.create(...)`**,
4. inspect a Dask-backed `xarray.DataArray`,
5. run a simple Africa vegetation example and an Asia coastal example,
6. adapt the same notebook to a different backend or endpoint.


## 1. Environment setup

Install the packages you need. For Google Earth Engine access, use the `ee` extra:

```bash
pip install "cubo[ee]" xarray dask matplotlib notebook
```

If you only want STAC access, `pip install cubo xarray dask matplotlib` is usually enough.


In [ ]:
import cubo
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import dask

# Optional: for the GEE examples
import ee


## 2. Earth Engine authentication

Run this once in a fresh environment if needed:

```python
ee.Authenticate()
```

Then initialize the high-volume endpoint used in the current `cubo` tutorial:

```python
ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com")
```


In [ ]:
# Uncomment if needed in a new environment:
# ee.Authenticate()
# ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com")


## 3. Why xarray + Dask?

`cubo.create(...)` returns an **`xarray.DataArray`**. That matters because you immediately get:

- labeled dimensions like `time`, `band`, `y`, and `x`,
- coordinate-aware selection with `.sel(...)` and `.isel(...)`,
- reductions like `.mean(...)`, `.median(...)`, `.groupby(...)`,
- optional **Dask-backed** lazy execution for larger-than-memory workflows.

A good beginner mental model is:

**API call → xarray object → lazy chunked operations → compute only when needed**


In [ ]:
toy = xr.DataArray(
    np.random.rand(6, 3, 32, 32),
    dims=("time", "band", "y", "x"),
    coords={
        "time": np.arange(6),
        "band": ["B1", "B2", "B3"]
    },
    name="toy_cube"
).chunk({"time": 2, "band": 1, "y": 32, "x": 32})

toy


In [ ]:
print("Dask-backed:", dask.is_dask_collection(toy.data))
print("Chunk structure:", toy.chunks)

lazy_result = toy.mean(("x", "y"))
lazy_result


The result above is still **lazy**. Nothing expensive happens until you ask for values explicitly with `.compute()`, `.load()`, plotting, or writing to disk.


In [ ]:
lazy_result.compute()


## 4. Common `cubo.create(...)` pattern

Most examples follow the same structure:

```python
cube = cubo.create(
    lat=...,
    lon=...,
    collection="...",
    bands=[...],
    start_date="YYYY-MM-DD",
    end_date="YYYY-MM-DD",
    edge_size=...,
    resolution=...,
    gee=True,              # optional
    stac="https://...",    # optional
)
```

Core decisions:
- **where**: `lat`, `lon`
- **what**: `collection`, `bands`
- **when**: `start_date`, `end_date`
- **how much**: `edge_size`, `resolution`
- **which backend**: default STAC, custom STAC, or `gee=True`


## 5. Example A — Africa vegetation monitoring with MODIS NDVI/EVI

A practical climate resilience example is to extract a small cube over the **Horn of Africa** or the **Sahel** and compute a time series of vegetation conditions.

This example uses the Google Earth Engine collection:

- `MODIS/061/MOD13Q1`
- 250 m pixels
- 16-day cadence
- bands such as `NDVI` and `EVI`

### Suggested study point
- Ethiopia / Horn of Africa: `lat=8.98`, `lon=39.79`


In [ ]:
africa_cube = cubo.create(
    lat=8.98,
    lon=39.79,
    collection="MODIS/061/MOD13Q1",
    bands=["NDVI", "EVI"],
    start_date="2021-01-01",
    end_date="2021-12-31",
    edge_size=64,
    resolution=250,
    gee=True,
)

africa_cube


### Inspect the object

Notice the labeled dimensions and the Dask-backed representation.


In [ ]:
print(africa_cube.dims)
print(africa_cube.coords)
print("Chunks:", africa_cube.chunks)


### Important note on scale factors

For MODIS vegetation indices, values are typically stored with a **scale factor**. A common first step is to convert them to approximately physical units.

For MOD13Q1, a simple teaching example is:

```python
africa_scaled = africa_cube / 10000.0
```

For production work, always verify the product documentation and quality flags.


In [ ]:
africa_scaled = africa_cube / 10000.0

africa_ts = (
    africa_scaled
    .sel(band="NDVI")
    .mean(("x", "y"))
)

africa_ts


In [ ]:
africa_ts.compute().plot(marker="o", figsize=(10, 4))
plt.title("Africa example: mean NDVI through time")
plt.ylabel("NDVI")
plt.grid(True, alpha=0.3)
plt.show()


### Discussion points for the live tutorial

- Why is the result still manageable even if the source is remote?
- Which step is lazy, and which step forces execution?
- How would you compare this year against a baseline climatology?
- Where would QA masking fit into the workflow?


## 6. Example B — Asia coastal monitoring with Sentinel-3 OLCI

For Asia, a simple medium-resolution example is to look at a coastal or delta region and build an RGB quicklook or a band composite from **Sentinel-3 OLCI**.

This example uses:

- `COPERNICUS/S3/OLCI`
- 21 spectral bands
- 300 m spatial resolution
- roughly 2-day revisit

### Suggested study point
- Ganges–Brahmaputra delta / coastal Bangladesh: `lat=22.20`, `lon=90.70`


In [ ]:
asia_cube = cubo.create(
    lat=22.20,
    lon=90.70,
    collection="COPERNICUS/S3/OLCI",
    bands=["Oa08_radiance", "Oa06_radiance", "Oa04_radiance"],
    start_date="2024-01-01",
    end_date="2024-01-10",
    edge_size=96,
    resolution=300,
    gee=True,
)

asia_cube


### Build a simple median RGB-style quicklook

The Earth Engine catalog example applies band-specific scale factors before visualization. For a notebook tutorial, it is enough to show the idea and note that **visualization scaling is dataset-specific**.


In [ ]:
asia_rgb = asia_cube.median("time")

asia_rgb


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, band in zip(axes, ["Oa08_radiance", "Oa06_radiance", "Oa04_radiance"]):
    asia_rgb.sel(band=band).compute().plot(ax=ax, add_colorbar=False)
    ax.set_title(band)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### Live discussion ideas

- Why is Sentinel-3 a good match for regional coastal monitoring?
- Which bands would you choose for turbidity, vegetation, or water color?
- What changes when you move from a quicklook to a quantitative indicator?


## 7. Optional: test a STAC endpoint with the same API

One nice feature of `cubo` is that the user-facing call stays almost the same when you switch from GEE to a STAC endpoint.

The documentation shows an Element84 Earth Search example like this:


In [ ]:
stac_cube = cubo.create(
    lat=4.31,
    lon=-76.20,
    collection="sentinel-s2-l2a-cogs",
    bands=["B05", "B06", "B07"],
    start_date="2020-01-01",
    end_date="2020-06-01",
    edge_size=128,
    resolution=20,
    stac="https://earth-search.aws.element84.com/v0",
)

stac_cube


This is a good section for workshop participants to experiment with:

- swapping the endpoint,
- changing the collection,
- changing bands and dates,
- comparing how the returned `xarray` object looks.


## 8. Minimal checklist for good Dask practice

When this notebook becomes a real workflow, the biggest beginner mistakes are usually about chunking and forcing computation too early.

### Good defaults for teaching
- keep the spatial subset small,
- start with a short date range,
- inspect `.chunks`,
- chain xarray operations before `.compute()`.

### Watch out for
- calling `.compute()` too soon,
- plotting very large arrays too early,
- using chunks that are too tiny,
- forgetting product-specific scaling or QA masks.


## 9. Export ideas

After a small analysis, common outputs are:

- PNG quicklooks,
- CSV time series,
- NetCDF/Zarr subsets,
- figures for reports or GitHub Pages.

For example:

```python
africa_ts.compute().to_series().to_csv("africa_ndvi_timeseries.csv")
```


## 10. Wrap-up

Key message:

> **Use `cubo` as the single entry point, `xarray` as the analysis model, and Dask as the scaling layer.**

That gives participants one consistent mental model:
1. request a remote EO cube,
2. inspect dimensions and chunks,
3. perform labeled operations lazily,
4. compute only for the final result.
